In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import joblib

from pathlib import Path

df = pd.read_csv("../data/student_mental_health.csv")

X = df.drop(columns=['Mental_Health_Risk', 'Student_ID'])
y = df['Mental_Health_Risk']



In [10]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

# --- Split 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# --- Transformer custom : IQRClipper ---
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        Q1 = np.percentile(X, 25, axis=0)
        Q3 = np.percentile(X, 75, axis=0)
        IQR = Q3 - Q1
        self.lower_ = Q1 - 1.5 * IQR
        self.upper_ = Q3 + 1.5 * IQR
        return self
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

# --- Colonnes ---
numerical_cols = [
    'Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week',
    'GPA', 'Social_Media_Hours', 'Screen_Time_Hours',
    'Financial_Stress', 'Family_Support', 'Attendance_Rate'
]
categorical_cols = [
    'Gender', 'Physical_Activity', 'Part_Time_Job',
    'Diet_Quality', 'Previous_Mental_Health_History'
]

# --- Pipeline complet ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('outliers', IQRClipper()),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# --- Fit sur Train, Transform sur Train et Test ---
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

# --- Sauvegarder les données prétraitées ---
processed_data = {
    'X_train': X_train_processed,
    'X_test':  X_test_processed,
    'y_train': y_train,
    'y_test':  y_test
}
joblib.dump(processed_data, 'preprocessed_data.pkl')
print('Données prétraitées sauvegardées avec succès !')
print('X_train_processed shape:', X_train_processed.shape)
print('X_test_processed shape: ', X_test_processed.shape)

Données prétraitées sauvegardées avec succès !
X_train_processed shape: (1600, 18)
X_test_processed shape:  (400, 18)
